In [1]:
#!/usr/bin/env python3
"""
Generate ChimeraX defattr file for B-factors from difference data.
Sums difference values by site and maps to sequential site numbering.
"""

import pandas as pd


In [6]:
def main():
    # Read the input files
    print("Reading input files...")
    diffs_df = pd.read_csv("../../results/summaries/tufted_duck_MHCII_binding.csv")

    # Filter to only rows where 293T entry >= -3
    diffs_df = diffs_df[diffs_df["entry in 293T cells"] >= -3]

    diff_col = (
        "entry difference between noSA tufted duck MHCII cells and SA26 SA23 cell mix"
    )

    # Sum the difference values by sequential_site
    print("\nSumming difference values by sequential_site...")
    site_sums = (
        diffs_df
        .groupby("sequential_site", as_index=False)[diff_col]
        .sum()
        .rename(columns={diff_col: "total_difference"})
    )

    # Generate the defattr file
    output_file = "bfactors_entry_difference_for_H7_sequential_site_numbering.defattr"
    print(f"\nWriting defattr file to {output_file}...")

    with open(output_file, "w") as f:
        # Write header
        f.write("# ChimeraX defattr file\n")
        f.write("# B-factors derived from summed difference values per sequential_site\n")
        f.write("#\n")
        f.write("attribute: difference\n")
        f.write("match mode: any\n")
        f.write("recipient: residues\n")
        f.write("\n")

        # Write data lines
        for _, row in site_sums.iterrows():
            site_value = str(row["sequential_site"])

            # Convert letters to uppercase for ChimeraX format, e.g. 125a -> 125A
            site_formatted = (
                site_value
                .replace("a", "A")
                .replace("b", "B")
                .replace("c", "C")
            )

            bfactor = float(row["total_difference"])
            f.write(f"\t:{site_formatted}\t{bfactor:.6f}\n")

    print(f"\nDefattr file successfully created: {output_file}")
    print(f"Total residues with B-factors: {len(site_sums)}")
    print("\nB-factor statistics:")
    print(f"  Min: {site_sums['total_difference'].min():.6f}")
    print(f"  Max: {site_sums['total_difference'].max():.6f}")
    print(f"  Mean: {site_sums['total_difference'].mean():.6f}")
    print(f"  Median: {site_sums['total_difference'].median():.6f}")
    print("\nExample lines from defattr file:")
    print(site_sums.head(5).to_string(index=False))


if __name__ == "__main__":
    main()

Reading input files...

Summing difference values by sequential_site...

Writing defattr file to bfactors_entry_difference_for_H7_sequential_site_numbering.defattr...

Defattr file successfully created: bfactors_entry_difference_for_H7_sequential_site_numbering.defattr
Total residues with B-factors: 506

B-factor statistics:
  Min: -17.262200
  Max: 64.101000
  Mean: 0.159133
  Median: -0.017250

Example lines from defattr file:
 sequential_site  total_difference
               1          -0.78330
               2          -0.14346
               3          -1.21420
               4           0.00000
               5          -2.92601
